# Specialist Training Research Workbench

This notebook is the inspection and experiment-control surface for electric guitar, strings, and wind/brass. The version-controlled Python scripts and YAML files remain authoritative. Paid cloud actions are never launched unless their `SUBMIT_*` guard is explicitly changed to `True`.

In [ ]:
from pathlib import Path
import sys

search_root = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (search_root, *search_root.parents)
        if (candidate / "workers/specialist_training_modal.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Repository root could not be located")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Audio, display

from research.workbench import (
    BASE_IDS,
    build_modal_command,
    display_command,
    experiment_snapshot,
    find_metric_json,
    load_index_manifest,
    local_listening_examples,
    modal_volume_paths,
    source_coverage,
    submit_modal_command,
    summarize_indexes,
)

sns.set_theme(context="notebook", style="whitegrid")
pd.set_option("display.max_colwidth", 120)
PROJECT_ROOT

## 1. Dataset contract

Confirm the exact indexed population before acquisition or training. Duplicate hashes are reported because the same audio must not silently inflate a target family.

In [ ]:
manifest = load_index_manifest()
index_summary = summarize_indexes()
display(index_summary)
print("Method:", manifest["method"])
print("Segment seconds:", manifest["segment_seconds"])
print("Profile:", manifest["profile"])

In [ ]:
family = "electric_guitar"
coverage = source_coverage(family)
display(coverage)
ax = coverage.plot(kind="bar", stacked=True, figsize=(13, 5), title=f"{family}: indexed audio by source")
ax.set_ylabel("Indexed audio files")
plt.tight_layout()
plt.show()

## 2. Local listening review

These are earlier local candidate/reference exports. They are useful for checking the playback workflow, but they are not evidence that the new three-model training has passed.

In [ ]:
examples = local_listening_examples(profile="clean-candidate-v1")
display(examples.head(20))

if not examples.empty:
    selected_audio = examples.iloc[0]["path"]
    print(selected_audio)
    display(Audio(filename=selected_audio))

## 3. Modal data status

A source receipt means its acquisition committed to the persistent Modal volume. It does not mean the complete index has been materialized.

In [ ]:
try:
    source_receipts = modal_volume_paths("/source_receipts")
    display(pd.DataFrame({"committed_source_receipt": source_receipts}))
except Exception as error:
    print(f"Modal status unavailable: {error}")

## 4. Resume cloud acquisition

Modal's asynchronous submission returns a persistent function-call ID immediately, so acquisition keeps running if this notebook, terminal, or network connection disappears. Review the command first. Change the guard only when ready.

In [ ]:
prepare_command = build_modal_command(action="prepare")
print(display_command(prepare_command))

SUBMIT_PREPARE = False
if SUBMIT_PREPARE:
    launch_receipt = submit_modal_command(
        prepare_command,
        confirm=True,
    )
    display(launch_receipt)

## 5. Reproducible training submission

Each target gets an independent BS-RoFormer checkpoint. The snapshot records the Git commit, dirty-worktree state, indexed-data hash, run ID, and exact detached command before any GPU is allocated.

In [ ]:
RUN_ID = "specialists-20260727"
STEPS = 200_000
EPOCHS = 1

training_commands = {}
for base_id in BASE_IDS:
    snapshot = experiment_snapshot(
        run_id=RUN_ID,
        base_id=base_id,
        steps=STEPS,
        epochs=EPOCHS,
    )
    training_commands[base_id] = build_modal_command(
        action="train",
        base_id=base_id,
        run_id=RUN_ID,
        steps=STEPS,
        epochs=EPOCHS,
    )
    print(base_id, snapshot["index_sha256"])
    print(" ", snapshot["command"])

SUBMIT_TRAINING = False
if SUBMIT_TRAINING:
    launches = {
        base_id: submit_modal_command(
            command,
            confirm=True,
        )
        for base_id, command in training_commands.items()
    }
    display(pd.DataFrame(launches).T)

## 6. Result discovery

After checkpoints and exports are downloaded, use this section to discover receipts/scorecards and compare predicted audio with its held-out reference. No model is promoted from file existence alone.

In [ ]:
metric_files = find_metric_json()
display(pd.DataFrame({"metric_or_receipt": [str(path.relative_to(PROJECT_ROOT)) for path in metric_files]}).tail(50))

## Research rule

A specialist checkpoint is accepted only after held-out objective evaluation and listening review. Dataset counts, successful training completion, or the existence of an audio file are not quality proof.